Pipeline:
    
    1. SFT teach how to answer
    2. Reward Model teach judge what is good
    3. PPO-RLHF practice with reward
    4. Evaluation + regression tests confirm real improvement

=> A base model learns **"how text usually continues on the internet"**; an assistant must learn **"how to respond responsibly to a user request."**

# Reward Model

In [3]:
import torch
import torch.nn as nn

In [55]:
class RewardModel(nn.Module):
    def __init__(self,base_model,hidden_dim=1024):
        super().__init__()
        self.base_model = base_model
        self.reward_head = nn.Linear(hidden_dim,1,dtype=base_model.dtype)
    def forward(self,input_ids):
        response_hs = self.base_model(input_ids,output_hidden_states=True).hidden_states[-1][:,-1] # [B,1]
        return self.reward_head(response_hs) # [,B]

In [53]:
def bradely_terry_loss(rm,chosen_ids,rejected_ids):
    chosen_reward = rm(chosen_ids)
    rejected_reward = rm(rejected_ids)
    loss = - torch.nn.functional.logsigmoid(chosen_reward - rejected_reward).mean()
    return loss

In [13]:
from transformers import AutoTokenizer,AutoModelForCausalLM
model_ids = "LiquidAI/LFM2.5-230M"
tokenizer = AutoTokenizer.from_pretrained(model_ids)
model = AutoModelForCausalLM.from_pretrained(model_ids)

Loading weights:   0%|          | 0/132 [00:00<?, ?it/s]

In [62]:
prompt = "what is 1+1? "
chosen = "the answer is 2"
rejected = "who knows"

In [63]:
chosen_ids = tokenizer.encode(prompt+chosen,return_tensors="pt")
rejected_ids = tokenizer.encode(prompt+rejected,return_tensors="pt")

In [64]:
rm = RewardModel(model)
rm(chosen_ids),rm(rejected_ids)

(tensor([[-0.5703]], dtype=torch.bfloat16, grad_fn=<AddmmBackward0>),
 tensor([[-0.9609]], dtype=torch.bfloat16, grad_fn=<AddmmBackward0>))

In [65]:
bradely_terry_loss(rm,chosen_ids,rejected_ids)

tensor(0.5156, dtype=torch.bfloat16, grad_fn=<NegBackward0>)